# infer-private-shard — sinh đáp án 1 checkpoint x 1 rotation trên 1 khoảng câu private (Kaggle T4)

Không chấm (private không có gold). Đọc `prompts.jsonl` đã pha sẵn (retrieval xong ở
job `prompt-v11-2x-hybrid`/rotation script), chỉ nạp Qwen+adapter rồi sinh. Ghi append +
resumable qua `da_co` — kernel bị giết giữa chừng thì chạy lại từ chỗ dở.

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 4, "commit": "", "noi_tiep": None,
         "output": "du_doan.jsonl", "checkpoint": "b920", "prompts_ten": "prompts.jsonl"}

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

SRC = next((c.parent.parent.parent for c in Path("/kaggle/input").glob("**/legalqa/bm25.py")
            if "legalqa-v10-src" in str(c)), None)
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/tmp/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
commit = (REPO / "COMMIT.txt").read_text().strip()
print("codebase:", commit, "| shard", SHARD)
assert not SHARD["commit"] or commit == SHARD["commit"], f"src {commit} != {SHARD['commit']}"

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
# Nối tiếp: dataset `orch-<job>-s<i>` mang phần đã sinh từ acc trước — copy về ĐÚNG tên
# output rồi infer_private_shard.py tự da_co() để biết id nào xong.
if SHARD["noi_tiep"]:
    for f in Path("/kaggle/input").glob("**/*.jsonl"):
        if SHARD["noi_tiep"] in str(f) and f.name == SHARD["output"]:
            shutil.copy(f, OUT / f.name)
            print(f"nối tiếp {f.name}: {sum(1 for _ in open(OUT / f.name, encoding='utf-8')):,} dòng")

In [ ]:
!pip -q install transformers==5.14.1 peft
!pip -q uninstall -y torchao

In [ ]:
PROMPTS = next(Path("/kaggle/input").glob(f"**/{SHARD['prompts_ten']}"), None)
assert PROMPTS, f"Không thấy {SHARD['prompts_ten']} — gắn dataset chứa file này"

BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
if [n for n in can if not (BASE / n).exists()]:
    FIX = Path("/tmp/base"); FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        if not (FIX / f.name).exists():
            (FIX / f.name).symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} index cần"
    for ten, that in zip(can, thuc):
        if not (FIX / ten).exists():
            (FIX / ten).symlink_to(that)
    BASE = FIX

ADAPTER = next((q.parent for q in Path("/kaggle/input").glob("**/adapter_config.json")
                if q.parent.name == SHARD["checkpoint"]), None)
assert ADAPTER, f"không thấy adapter '{SHARD['checkpoint']}' — gắn dataset legalqa-ckpt-v11"
print("checkpoint", SHARD["checkpoint"], "| base", BASE, "| adapter", ADAPTER, "| prompts", PROMPTS)

In [ ]:
ra = OUT / SHARD["output"]
subprocess.run([sys.executable, "scripts/infer_private_shard.py",
    "--model", str(BASE), "--adapter", str(ADAPTER), "--prompts", str(PROMPTS),
    "--ra", str(ra), "--tu", str(SHARD["tu"]), "--den", str(SHARD["den"]),
    "--max-new", str(SHARD.get("max_new", 1024)), "--gen-bs", str(SHARD.get("gen_bs", 2)),
    "--gio-toi-da", str(SHARD.get("gio_toi_da", 8.5)),
], check=True)

from infer import da_co
# Kiểm đủ câu của khoảng [tu, den) rồi mới ghi mốc — như shard_prompt.ipynb.
rows_khoang = [json.loads(l) for l in open(PROMPTS, encoding="utf-8")][SHARD["tu"]:SHARD["den"]]
can = {r["id"] for r in rows_khoang}
co = can & da_co(ra)
from kaggle_orchestra import ghi_moc
if co == can:
    print("mốc:", ghi_moc(ra, SHARD["tu"], SHARD["den"]), flush=True)
else:
    print(f"CHƯA XONG shard: {len(co)}/{len(can)} câu — không ghi mốc", flush=True)